<a href="https://colab.research.google.com/github/sigma-asif/explainable-transport-mode-choice/blob/main/3_sensitivity_excluding_context_variables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, shutil
from google.colab import drive

if os.path.ismount("/content/drive"):
    print("Drive আগে থেকেই mounted আছে, কিছু মোছা হয়নি।")
else:
    shutil.rmtree("/content/drive", ignore_errors=True)  # শুধু নকল local folder মুছছে
    drive.mount("/content/drive")

Drive আগে থেকেই mounted আছে, কিছু মোছা হয়নি।


In [ ]:
"""
ABLATION: remove the five context-specific Mode_When_* / Mode_In_Weather variables
==============================================================================
Run in Google Colab AFTER you have run the main notebook once (it saves
model_matrix_43_features.csv). This script is standalone: it does not need the
main notebook's functions, it only reads that saved CSV.

Variants
--------
FULL            : 43 columns (sanity check, must reproduce your frozen results)
A_no_context5   : drop all 15 dummies of Depressed, Happy, Stressed, Urgent, Weather
B_no_psych3     : drop only Depressed, Happy, Stressed (the "Psychological" role)

Same protocol as the paper: stratified 5-fold (seed 42), RandomOverSampler on
training folds only, SVM/MNL scaled after oversampling, TreeSHAP on held-out
observations only, Local Top-2 dominant role.
"""

import os
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, cohen_kappa_score, matthews_corrcoef,
                             roc_auc_score, precision_recall_fscore_support)
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.svm import SVC
from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SETTINGS (edit paths only)
# ---------------------------------------------------------------------------
MODEL_MATRIX_CSV = ("/content/drive/MyDrive/Research/Transport Mode/FINAL TRANSPORT MODE/"
                    "Q1 PIPELINE V2/preprocessing/model_matrix_43_features.csv")
OUT_DIR = ("/content/drive/MyDrive/Research/Transport Mode/FINAL TRANSPORT MODE/"
           "Q1 PIPELINE V2/ablation")
TARGET = "Transport_Grouped"
SEED = 42
K = 5
N_REPEATS = 10
N_EST = 100
os.makedirs(OUT_DIR, exist_ok=True)

ALL_CONTEXT = ["Mode_When_Depressed", "Mode_When_Happy", "Mode_When_Stressed",
               "Mode_When_Urgent", "Mode_In_Weather"]
PSYCH3 = ["Mode_When_Depressed", "Mode_When_Happy", "Mode_When_Stressed"]

ECONOMIC = {"Monthly Income", "Average Expendature for transportation",
            "Importance of Cost in Mode Choice", "Effect of Cost on Mode Choice"}
SAFETY = {"Importance of Safety in Mode Choice", "Mobility Barrier on Mode Choice",
          "Accident Memories On Mode Choice", "Trust level based on safety in Bus",
          "Trust level based on safety in CNG", "Trust level based on safety in Easybike",
          "Trust level based on safety in Rickshaw", "Trust level based on safety in Bike",
          "Trust level based on safety in Car", "Trust level based on safety in Cycle"}
PSYCH = set(PSYCH3)


def parent_of(col):
    """Mode_When_Happy__Public -> Mode_When_Happy; other columns unchanged."""
    return col.split("__")[0] if "__" in col else col


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
df = pd.read_csv(MODEL_MATRIX_CSV)
assert TARGET in df.columns, f"{TARGET} not found in {MODEL_MATRIX_CSV}"
ALL_FEATS = [c for c in df.columns if c != TARGET]
assert len(ALL_FEATS) == 43, f"Expected 43 predictors, found {len(ALL_FEATS)}"

le = LabelEncoder()
y = pd.Series(le.fit_transform(df[TARGET]), name=TARGET)
class_names = list(le.classes_)
classes = list(range(len(class_names)))


def feats_without(parents):
    return [c for c in ALL_FEATS if parent_of(c) not in parents]


VARIANTS = {
    "FULL": ALL_FEATS,
    "A_no_context5": feats_without(ALL_CONTEXT),
    "B_no_psych3": feats_without(PSYCH3),
}
for name, cols in VARIANTS.items():
    print(f"{name:15s}: {len(cols)} encoded columns")


# ---------------------------------------------------------------------------
# Helpers (identical logic to the frozen pipeline)
# ---------------------------------------------------------------------------
def make_models():
    return {
        "RF": RandomForestClassifier(n_estimators=N_EST, random_state=SEED, n_jobs=-1),
        "SVM": SVC(kernel="rbf", probability=True, random_state=SEED),
        "MNL": LogisticRegression(solver="lbfgs", max_iter=2000, random_state=SEED),
        "XGBoost": XGBClassifier(n_estimators=N_EST, eval_metric="mlogloss",
                                 random_state=SEED, verbosity=0, n_jobs=-1),
        "LightGBM": LGBMClassifier(n_estimators=N_EST, random_state=SEED,
                                   verbose=-1, n_jobs=-1),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=N_EST,
                                                       random_state=SEED),
    }


SCALED = {"SVM", "MNL"}


def fit_fold(name, est, Xtr, ytr, Xte):
    Xr, yr = RandomOverSampler(random_state=SEED).fit_resample(Xtr, ytr)
    if name in SCALED:
        sc = StandardScaler()
        Xr_f, Xte_f = sc.fit_transform(Xr), sc.transform(Xte)
    else:
        Xr_f, Xte_f = Xr, Xte
    m = clone(est).fit(Xr_f, yr)
    return m, m.predict(Xte_f), m.predict_proba(Xte_f), Xte_f


def metrics(yt, yp, pr):
    yb = label_binarize(yt, classes=classes)
    auc = roc_auc_score(yb, pr, average="macro", multi_class="ovr")
    _, _, f1, _ = precision_recall_fscore_support(yt, yp, labels=classes,
                                                  average=None, zero_division=0)
    return {"accuracy": accuracy_score(yt, yp), "kappa": cohen_kappa_score(yt, yp),
            "mcc": matthews_corrcoef(yt, yp), "macro_auc": auc,
            "macro_f1": float(np.mean(f1))}


def norm_shap(sv, n, p, c):
    if isinstance(sv, list):
        return np.stack([np.asarray(v) for v in sv], axis=2)
    a = np.asarray(sv)
    if a.shape == (n, p, c):
        return a
    if a.shape == (c, n, p):
        return np.transpose(a, (1, 2, 0))
    raise ValueError(f"Unexpected SHAP shape {a.shape}")


# ---------------------------------------------------------------------------
# 1) Six-model fixed 5-fold OOF screen
# ---------------------------------------------------------------------------
def six_model_screen(X):
    cv = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
    models = make_models()
    pred = {m: np.full(len(X), -1) for m in models}
    prob = {m: np.zeros((len(X), len(classes))) for m in models}
    for tr, te in cv.split(X, y):
        for m, est in models.items():
            _, p, pr, _ = fit_fold(m, est, X.iloc[tr], y.iloc[tr], X.iloc[te])
            pred[m][te], prob[m][te] = p, pr
    rows = [{"model": m, **metrics(y, pred[m], prob[m])} for m in models]
    return pd.DataFrame(rows).sort_values("macro_f1", ascending=False)


# ---------------------------------------------------------------------------
# 2) Repeated RF vs LightGBM, 5-fold x 10 (repeat-level OOF metrics)
# ---------------------------------------------------------------------------
def repeated_rf_lgbm(X):
    models = {"RF": make_models()["RF"], "LightGBM": make_models()["LightGBM"]}
    rskf = RepeatedStratifiedKFold(n_splits=K, n_repeats=N_REPEATS, random_state=SEED)
    store = {m: {r: {"p": np.full(len(X), -1), "pr": np.zeros((len(X), len(classes)))}
                 for r in range(N_REPEATS)} for m in models}
    for i, (tr, te) in enumerate(rskf.split(X, y)):
        r = i // K
        for m, est in models.items():
            _, p, pr, _ = fit_fold(m, est, X.iloc[tr], y.iloc[tr], X.iloc[te])
            store[m][r]["p"][te], store[m][r]["pr"][te] = p, pr
    rows = []
    for m in models:
        for r in range(N_REPEATS):
            rows.append({"model": m, "repeat": r + 1,
                         **metrics(y, store[m][r]["p"], store[m][r]["pr"])})
    rep = pd.DataFrame(rows)
    summ = rep.groupby("model")[["accuracy", "mcc", "macro_auc", "macro_f1"]] \
              .agg(["mean", "std"]).round(4)
    return rep, summ


# ---------------------------------------------------------------------------
# 3) Final RF: fixed 5-fold OOF + held-out SHAP + conceptual aggregation + roles
# ---------------------------------------------------------------------------
def final_rf_shap_roles(X, cols):
    cv = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
    rf = make_models()["RF"]
    oof_p = np.full(len(X), -1)
    oof_pr = np.zeros((len(X), len(classes)))
    sv_enc = np.zeros((len(X), len(cols), len(classes)))
    for tr, te in cv.split(X, y):
        m, p, pr, Xte = fit_fold("RF", rf, X.iloc[tr], y.iloc[tr], X.iloc[te])
        oof_p[te], oof_pr[te] = p, pr
        raw = shap.TreeExplainer(m).shap_values(Xte)
        sv_enc[te] = norm_shap(raw, len(te), len(cols), len(classes))
    overall = metrics(y, oof_p, oof_pr)

    # Sum SIGNED SHAP over the dummies of each Mode_When_* parent -> conceptual level
    concept = list(dict.fromkeys(parent_of(c) for c in cols))
    cidx = {f: j for j, f in enumerate(concept)}
    sv = np.zeros((len(X), len(concept), len(classes)))
    for j, c in enumerate(cols):
        sv[:, cidx[parent_of(c)], :] += sv_enc[:, j, :]

    global_imp = (pd.Series(np.abs(sv).mean(axis=(0, 2)), index=concept)
                  .sort_values(ascending=False))

    # Roles restricted to features still present; empty roles are dropped
    other = set(concept) - ECONOMIC - SAFETY - PSYCH
    roles = {"Economic": ECONOMIC & set(concept), "Safety": SAFETY & set(concept),
             "Psychological": PSYCH & set(concept), "Other": other}
    roles = {r: f for r, f in roles.items() if len(f) > 0}

    sens = []
    top2_counts = None
    for k in [1, 2, 3]:
        cnt = defaultdict(int)
        for i in range(len(X)):
            local = sv[i, :, oof_p[i]]
            scores = {r: sum(sorted((abs(local[cidx[f]]) for f in fs), reverse=True)[:k])
                      for r, fs in roles.items()}
            cnt[max(scores, key=scores.get)] += 1
        sens.append({"k": k, **{r: cnt.get(r, 0) for r in roles}})
        if k == 2:
            top2_counts = dict(cnt)
    return overall, global_imp, pd.DataFrame(sens), top2_counts


# ---------------------------------------------------------------------------
# RUN ALL VARIANTS
# ---------------------------------------------------------------------------
summary_rows, role_rows, rank_tables = [], [], {}

for vname, cols in VARIANTS.items():
    print("\n" + "=" * 90)
    print(f"VARIANT: {vname}  ({len(cols)} columns)")
    print("=" * 90)
    X = df[cols].copy()

    six = six_model_screen(X)
    six.to_csv(os.path.join(OUT_DIR, f"{vname}_six_model_oof.csv"), index=False)
    print("\nSix-model fixed 5-fold OOF:")
    print(six.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    rep, rep_summ = repeated_rf_lgbm(X)
    rep.to_csv(os.path.join(OUT_DIR, f"{vname}_repeated_rf_lgbm.csv"), index=False)
    print("\nRepeated 5x10 (mean, sd):")
    print(rep_summ.to_string())

    overall, gimp, sens, top2 = final_rf_shap_roles(X, cols)
    gimp.to_csv(os.path.join(OUT_DIR, f"{vname}_global_shap_conceptual.csv"),
                header=["mean_abs_shap"])
    sens.to_csv(os.path.join(OUT_DIR, f"{vname}_role_topk_sensitivity.csv"), index=False)
    rank_tables[vname] = gimp

    print("\nFinal RF fixed OOF:", {k: round(v, 4) for k, v in overall.items()})
    print("\nTop-10 conceptual SHAP:")
    print(gimp.head(10).round(4).to_string())
    print("\nLocal Top-2 dominant role counts:", top2,
          "| Economic share = "
          f"{100 * top2.get('Economic', 0) / len(X):.1f}%")
    print("\nTop-k sensitivity:")
    print(sens.to_string(index=False))

    rf_rep = rep[rep.model == "RF"][["accuracy", "macro_f1"]].mean()
    summary_rows.append({
        "variant": vname, "n_columns": len(cols),
        "RF_fixed_accuracy": overall["accuracy"], "RF_fixed_macroF1": overall["macro_f1"],
        "RF_fixed_macroAUC": overall["macro_auc"], "RF_fixed_MCC": overall["mcc"],
        "RF_rep10_accuracy": rf_rep["accuracy"], "RF_rep10_macroF1": rf_rep["macro_f1"],
        "Economic_share_%": 100 * top2.get("Economic", 0) / len(X),
        "Safety_share_%": 100 * top2.get("Safety", 0) / len(X),
        "Psych_share_%": 100 * top2.get("Psychological", 0) / len(X),
        "Other_share_%": 100 * top2.get("Other", 0) / len(X),
        "Top3_SHAP": " > ".join(gimp.index[:3]),
    })

print("\n" + "#" * 90)
print("ABLATION SUMMARY (put this in the paper)")
print("#" * 90)
summary = pd.DataFrame(summary_rows)
print(summary.round(4).T.to_string())
summary.to_csv(os.path.join(OUT_DIR, "ablation_summary.csv"), index=False)
print(f"\nSaved all outputs in: {OUT_DIR}")
print("\nSANITY CHECK: FULL must give accuracy 0.8879, macro-F1 0.8547, AUC 0.9471, "
      "Economic 217/348 (62.4%). If not, the CSV or settings differ from the frozen run.")

FULL           : 43 encoded columns
A_no_context5  : 28 encoded columns
B_no_psych3    : 34 encoded columns

VARIANT: FULL  (43 columns)

Six-model fixed 5-fold OOF:
           model  accuracy  kappa    mcc  macro_auc  macro_f1
              RF    0.8879 0.7720 0.7794     0.9471    0.8547
         XGBoost    0.8678 0.7450 0.7454     0.9255    0.8142
        LightGBM    0.8678 0.7325 0.7381     0.9298    0.8135
             SVM    0.8477 0.6959 0.6988     0.9321    0.8104
GradientBoosting    0.8448 0.6980 0.6986     0.9094    0.7765
             MNL    0.7270 0.5211 0.5285     0.8651    0.6757

Repeated 5x10 (mean, sd):
         accuracy             mcc         macro_auc         macro_f1        
             mean     std    mean     std      mean     std     mean     std
model                                                                       
LightGBM   0.8670  0.0189  0.7366  0.0384    0.9349  0.0069   0.8043  0.0317
RF         0.8848  0.0130  0.7731  0.0272    0.9486  0.0062   0.8

In [ ]:
import pandas as pd
p = "/content/drive/MyDrive/Research/Transport Mode/FINAL TRANSPORT MODE/Q1 PIPELINE V2/ablation/A_no_context5_repeated_rf_lgbm.csv"
d = pd.read_csv(p).pivot(index="repeat", columns="model", values="macro_f1")
diff = d["RF"] - d["LightGBM"]
print("RF better in", (diff > 0).sum(), "of", len(diff), "repeats; mean diff =", round(diff.mean(), 4))

RF better in 7 of 10 repeats; mean diff = 0.0105


In [ ]:
import pandas as pd, numpy as np
from scipy.stats import t

p = "/content/drive/MyDrive/Research/Transport Mode/FINAL TRANSPORT MODE/Q1 PIPELINE V2/ablation/A_no_context5_repeated_rf_lgbm.csv"
d = pd.read_csv(p)
for metric in ["accuracy", "mcc", "macro_auc", "macro_f1"]:
    w = d.pivot(index="repeat", columns="model", values=metric)
    diff = (w["RF"] - w["LightGBM"]).values
    m, s, n = diff.mean(), diff.std(ddof=1), len(diff)
    h = t.ppf(0.975, n - 1) * s / np.sqrt(n)
    print(f"{metric:10s} mean diff {m:.4f}  95% CI [{m-h:.4f}, {m+h:.4f}]  RF better {(diff>0).sum()}/10")

accuracy   mean diff 0.0106  95% CI [0.0008, 0.0204]  RF better 7/10
mcc        mean diff 0.0192  95% CI [-0.0000, 0.0385]  RF better 7/10
macro_auc  mean diff 0.0115  95% CI [0.0063, 0.0166]  RF better 10/10
macro_f1   mean diff 0.0105  95% CI [-0.0052, 0.0263]  RF better 7/10


In [ ]:
import pandas as pd, numpy as np
from scipy.stats import t

p = "/content/drive/MyDrive/Research/Transport Mode/FINAL TRANSPORT MODE/Q1 PIPELINE V2/ablation/B_no_psych3_repeated_rf_lgbm.csv"
d = pd.read_csv(p)
for metric in ["accuracy", "mcc", "macro_auc", "macro_f1"]:
    w = d.pivot(index="repeat", columns="model", values=metric)
    diff = (w["RF"] - w["LightGBM"]).values
    m, s, n = diff.mean(), diff.std(ddof=1), len(diff)
    h = t.ppf(0.975, n - 1) * s / np.sqrt(n)
    print(f"{metric:10s} mean diff {m:.4f}  95% CI [{m-h:.4f}, {m+h:.4f}]  RF better {(diff>0).sum()}/10")

accuracy   mean diff 0.0103  95% CI [0.0016, 0.0191]  RF better 8/10
mcc        mean diff 0.0198  95% CI [0.0022, 0.0373]  RF better 8/10
macro_auc  mean diff 0.0154  95% CI [0.0101, 0.0206]  RF better 10/10
macro_f1   mean diff 0.0091  95% CI [-0.0075, 0.0256]  RF better 7/10


In [ ]:
# ============================================================
# PER-CLASS ABLATION ANALYSIS
# Fixed 5-fold OOF Random Forest
# Same protocol as main / ablation analysis
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    accuracy_score
)

# ------------------------------------------------------------
# 1. Function: fixed 5-fold OOF RF predictions
# ------------------------------------------------------------
def fixed_oof_rf_predictions(X):
    """
    Reproduce the paper's fixed 5-fold OOF Random Forest evaluation:
    - StratifiedKFold
    - shuffle=True
    - random_state=42
    - RandomOverSampler only on training fold
    - untouched held-out fold
    """

    cv = StratifiedKFold(
        n_splits=K,
        shuffle=True,
        random_state=SEED
    )

    rf = make_models()["RF"]

    oof_pred = np.full(len(X), -1, dtype=int)
    oof_prob = np.zeros((len(X), len(classes)))

    for fold, (tr, te) in enumerate(cv.split(X, y), start=1):

        _, pred, prob, _ = fit_fold(
            "RF",
            rf,
            X.iloc[tr],
            y.iloc[tr],
            X.iloc[te]
        )

        oof_pred[te] = pred
        oof_prob[te] = prob

    assert np.all(oof_pred >= 0), "Some OOF predictions are missing."

    return oof_pred, oof_prob


# ------------------------------------------------------------
# 2. Run per-class metrics for FULL + ablation variants
# ------------------------------------------------------------
all_class_rows = []
all_confusion = {}

for vname, cols in VARIANTS.items():

    print("\n" + "=" * 80)
    print(f"VARIANT: {vname}")
    print(f"Encoded predictors: {len(cols)}")
    print("=" * 80)

    X = df[cols].copy()

    # Fixed OOF RF predictions
    oof_pred, oof_prob = fixed_oof_rf_predictions(X)

    # Per-class metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        y,
        oof_pred,
        labels=classes,
        average=None,
        zero_division=0
    )

    rows = []

    for c, cname in zip(classes, class_names):

        row = {
            "variant": vname,
            "class": cname,
            "precision": precision[c],
            "recall": recall[c],
            "f1": f1[c],
            "support": int(support[c])
        }

        rows.append(row)
        all_class_rows.append(row)

    per_class_df = pd.DataFrame(rows)

    print("\nPer-class performance:")
    print(
        per_class_df[
            ["class", "precision", "recall", "f1", "support"]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------
    cm = confusion_matrix(
        y,
        oof_pred,
        labels=classes
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual_{x}" for x in class_names],
        columns=[f"Pred_{x}" for x in class_names]
    )

    all_confusion[vname] = cm_df

    print("\nConfusion matrix:")
    print(cm_df.to_string())

    # Save individual outputs
    per_class_df.to_csv(
        os.path.join(
            OUT_DIR,
            f"{vname}_RF_per_class_metrics.csv"
        ),
        index=False
    )

    cm_df.to_csv(
        os.path.join(
            OUT_DIR,
            f"{vname}_RF_confusion_matrix.csv"
        )
    )


# ------------------------------------------------------------
# 3. Combined per-class results
# ------------------------------------------------------------
class_results = pd.DataFrame(all_class_rows)

class_results.to_csv(
    os.path.join(
        OUT_DIR,
        "ablation_RF_per_class_all_variants.csv"
    ),
    index=False
)

print("\n\n" + "#" * 90)
print("ALL PER-CLASS RESULTS")
print("#" * 90)

print(
    class_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 4. PUBLIC CLASS — main comparison
# ------------------------------------------------------------
public_results = (
    class_results[
        class_results["class"].astype(str).str.lower() == "public"
    ]
    .copy()
    .reset_index(drop=True)
)

if len(public_results) == 0:
    raise ValueError(
        f"'Public' class not found. Available classes: {class_names}"
    )

# FULL model baseline
full_public = public_results[
    public_results["variant"] == "FULL"
].iloc[0]

public_results["recall_change_vs_FULL"] = (
    public_results["recall"] - full_public["recall"]
)

public_results["f1_change_vs_FULL"] = (
    public_results["f1"] - full_public["f1"]
)

public_results["precision_change_vs_FULL"] = (
    public_results["precision"] - full_public["precision"]
)

print("\n\n" + "#" * 90)
print("PUBLIC CLASS COMPARISON")
print("#" * 90)

print(
    public_results[
        [
            "variant",
            "precision",
            "recall",
            "f1",
            "support",
            "recall_change_vs_FULL",
            "f1_change_vs_FULL"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

public_results.to_csv(
    os.path.join(
        OUT_DIR,
        "ablation_Public_class_comparison.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 5. Publication-friendly wide table
# ------------------------------------------------------------
publication_table = class_results.pivot(
    index="class",
    columns="variant",
    values=["precision", "recall", "f1"]
)

publication_table.to_csv(
    os.path.join(
        OUT_DIR,
        "ablation_per_class_publication_table.csv"
    )
)

print("\n\nPublication-style wide table:")
print(publication_table.round(4).to_string())


# ------------------------------------------------------------
# 6. Short manuscript-ready Public summary
# ------------------------------------------------------------
print("\n\n" + "#" * 90)
print("PUBLIC RECALL — MANUSCRIPT NUMBERS")
print("#" * 90)

for _, r in public_results.iterrows():

    print(
        f"{r['variant']:15s} | "
        f"Precision = {r['precision']:.4f} | "
        f"Recall = {r['recall']:.4f} | "
        f"F1 = {r['f1']:.4f} | "
        f"Support = {int(r['support'])} | "
        f"Δ Recall vs FULL = {r['recall_change_vs_FULL']:+.4f}"
    )

print("\nSaved files in:")
print(OUT_DIR)


VARIANT: FULL
Encoded predictors: 43

Per-class performance:
            class  precision  recall     f1  support
     Intermediate     0.8805  0.9693 0.9228      228
Low-cost Personal     0.9375  0.8108 0.8696       37
    Private/Hired     0.9500  0.8261 0.8837       23
           Public     0.8667  0.6500 0.7429       60

Confusion matrix:
                          Pred_Intermediate  Pred_Low-cost Personal  Pred_Private/Hired  Pred_Public
Actual_Intermediate                     221                       0                   1            6
Actual_Low-cost Personal                  7                      30                   0            0
Actual_Private/Hired                      3                       1                  19            0
Actual_Public                            20                       1                   0           39

VARIANT: A_no_context5
Encoded predictors: 28

Per-class performance:
            class  precision  recall     f1  support
     Intermediate     0.8